# ETF Holdings and Exposure Metrics

This notebook calculates holdings-based risk measures, including number of holdings, sector concentration, country concentration and currency risk. These measures complement the historical market-risk metrics by examining what each ETF actually holds.

In [1]:
import pandas as pd

etf_universe = pd.read_csv(
    "../data/processed/etf_universe_clean.csv"
)

holdings_columns = [
    "ETF_ID",
    "ETF_Name",
    "Ticker",
    "Asset_Class",
    "Sector_Focus",
    "Number_of_Holdings",
    "Holdings_Data_Quality",
    "Main_Currency_Exposure",
    "Currency_Risk_Level"
]

holdings_data = etf_universe[holdings_columns].copy()

print("Rows:", holdings_data.shape[0])
print("Columns:", holdings_data.shape[1])

holdings_data.head()

Rows: 20
Columns: 9


,ETF_ID,ETF_Name,Ticker,Asset_Class,Sector_Focus,Number_of_Holdings,Holdings_Data_Quality,Main_Currency_Exposure,Currency_Risk_Level
0,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,Broad,3782.0,Exact provider,USD-dominant / multi-currency,Medium
1,SWDA,iShares Core MSCI World UCITS ETF,SWDA.L,Equity,Broad,1283.0,Exact provider,USD-dominant / multi-currency developed markets,Medium
2,VUAG,Vanguard S&P 500 UCITS ETF,VUAG.L,Equity,Broad large-cap,504.0,Exact provider,USD,Medium
3,ISF,iShares Core FTSE 100 UCITS ETF,ISF.L,Equity,Broad large-cap,100.0,Exact provider,GBP-dominant,Low
4,EQQQ,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L,Equity,Growth / Nasdaq,100.0,Benchmark-level,USD,Medium


In [2]:
print("Missing values:")
print(holdings_data.isna().sum())

print("\nNumber of holdings data type:")
print(holdings_data["Number_of_Holdings"].dtype)

print("\nCurrency risk categories:")
print(holdings_data["Currency_Risk_Level"].value_counts(dropna=False))

Missing values:
ETF_ID                    0
ETF_Name                  0
Ticker                    0
Asset_Class               0
Sector_Focus              0
Number_of_Holdings        1
Holdings_Data_Quality     0
Main_Currency_Exposure    0
Currency_Risk_Level       0
dtype: int64

Number of holdings data type:
float64

Currency risk categories:
Currency_Risk_Level
Medium    13
Low        4
High       3
Name: count, dtype: int64


In [3]:
holdings_data.loc[
    holdings_data["Number_of_Holdings"].isna(),
    ["ETF_ID", "ETF_Name", "Ticker"]
]

,ETF_ID,ETF_Name,Ticker
19,IWDP,iShares Developed Markets Property Yield UCITS...,IWDP.L


In [4]:
etf_universe.loc[
    etf_universe["Ticker"] == "IWDP.L",
    "Number_of_Holdings"
] = 315

etf_universe.loc[
    etf_universe["Ticker"] == "IWDP.L",
    "Data_As_Of"
] = "2026-07-21"

etf_universe["Number_of_Holdings"] = (
    pd.to_numeric(
        etf_universe["Number_of_Holdings"],
        errors="coerce"
    ).astype("Int64")
)

holdings_data = etf_universe[holdings_columns].copy()

print(
    "Missing holdings figures:",
    holdings_data["Number_of_Holdings"].isna().sum()
)

holdings_data.loc[
    holdings_data["Ticker"] == "IWDP.L"
]

Missing holdings figures: 0


,ETF_ID,ETF_Name,Ticker,Asset_Class,Sector_Focus,Number_of_Holdings,Holdings_Data_Quality,Main_Currency_Exposure,Currency_Risk_Level
19,IWDP,iShares Developed Markets Property Yield UCITS...,IWDP.L,Property,Property / REITs,315,Provider value not visible in scraped page,USD / multi-developed property currencies,Medium


In [5]:
etf_universe.to_csv(
    "../data/processed/etf_universe_clean.csv",
    index=False
)

print("Updated ETF universe saved.")
print(
    "Missing holdings figures:",
    etf_universe["Number_of_Holdings"].isna().sum()
)

Updated ETF universe saved.
Missing holdings figures: 0


In [6]:
import numpy as np

holdings_data["Log_Number_of_Holdings"] = np.log1p(
    holdings_data["Number_of_Holdings"]
)

minimum_holdings = holdings_data["Log_Number_of_Holdings"].min()
maximum_holdings = holdings_data["Log_Number_of_Holdings"].max()

holdings_data["Holdings_Count_Risk"] = (
    100
    * (
        maximum_holdings
        - holdings_data["Log_Number_of_Holdings"]
    )
    / (
        maximum_holdings
        - minimum_holdings
    )
).round(2)

holdings_data[
    [
        "ETF_Name",
        "Ticker",
        "Number_of_Holdings",
        "Holdings_Count_Risk"
    ]
].sort_values(
    "Holdings_Count_Risk",
    ascending=False
)

,ETF_Name,Ticker,Number_of_Holdings,Holdings_Count_Risk
18,iShares Physical Gold ETC,SGLN.L,1,100.0
10,iShares S&P 500 Health Care Sector UCITS ETF,IUHC.L,60,62.89
12,Vanguard U.K. Gilt UCITS ETF,VGOV.L,64,62.2
9,iShares S&P 500 Information Technology Sector ...,IUIT.L,74,60.64
14,iShares $ Treasury Bond 1-3yr UCITS ETF GBP He...,IBTG.L,91,58.43
3,iShares Core FTSE 100 UCITS ETF,ISF.L,100,57.41
4,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L,100,57.41
11,iShares Global Clean Energy Transition UCITS ETF,INRG.L,106,56.79
19,iShares Developed Markets Property Yield UCITS...,IWDP.L,315,45.03
6,Vanguard FTSE Japan UCITS ETF,VJPN.L,476,40.55


In [7]:
currency_risk_mapping = {
    "Low": 25,
    "Medium": 50,
    "High": 75
}

holdings_data["Currency_Risk_Score"] = (
    holdings_data["Currency_Risk_Level"]
    .map(currency_risk_mapping)
)

holdings_data[
    [
        "ETF_Name",
        "Ticker",
        "Main_Currency_Exposure",
        "Currency_Risk_Level",
        "Currency_Risk_Score"
    ]
].sort_values(
    "Currency_Risk_Score",
    ascending=False
)

,ETF_Name,Ticker,Main_Currency_Exposure,Currency_Risk_Level,Currency_Risk_Score
11,iShares Global Clean Energy Transition UCITS ETF,INRG.L,USD / multi-country,High,75
7,Vanguard FTSE Emerging Markets UCITS ETF,VFEG.L,TWD/CNY/INR/BRL/ZAR/SAR/MXN/multi-EM,High,75
8,iShares MSCI China UCITS ETF,ICHN.AS,CNY/HKD/USD,High,75
0,Vanguard FTSE All-World UCITS ETF,VWRP.L,USD-dominant / multi-currency,Medium,50
1,iShares Core MSCI World UCITS ETF,SWDA.L,USD-dominant / multi-currency developed markets,Medium,50
18,iShares Physical Gold ETC,SGLN.L,USD gold price exposure,Medium,50
16,iShares Global Corp Bond UCITS ETF,CORP.L,USD / multi-currency global corporate bonds,Medium,50
15,iShares Core Global Aggregate Bond UCITS ETF,AGGG.L,USD / multi-currency global aggregate bonds,Medium,50
13,iShares Global Govt Bond UCITS ETF,IGLO.L,USD / multi-developed government bond currencies,Medium,50
10,iShares S&P 500 Health Care Sector UCITS ETF,IUHC.L,USD,Medium,50


In [8]:
print(
    "Missing currency scores:",
    holdings_data["Currency_Risk_Score"].isna().sum()
)

holdings_metrics_base = holdings_data[
    [
        "ETF_ID",
        "ETF_Name",
        "Ticker",
        "Asset_Class",
        "Number_of_Holdings",
        "Holdings_Count_Risk",
        "Main_Currency_Exposure",
        "Currency_Risk_Level",
        "Currency_Risk_Score"
    ]
].copy()

holdings_metrics_base.to_csv(
    "../data/processed/etf_holdings_metrics_base.csv",
    index=False
)

print("Base holdings metrics saved.")
holdings_metrics_base.head()

Missing currency scores: 0
Base holdings metrics saved.


,ETF_ID,ETF_Name,Ticker,Asset_Class,Number_of_Holdings,Holdings_Count_Risk,Main_Currency_Exposure,Currency_Risk_Level,Currency_Risk_Score
0,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,3782,18.07,USD-dominant / multi-currency,Medium,50
1,SWDA,iShares Core MSCI World UCITS ETF,SWDA.L,Equity,1283,29.8,USD-dominant / multi-currency developed markets,Medium,50
2,VUAG,Vanguard S&P 500 UCITS ETF,VUAG.L,Equity,504,39.94,USD,Medium,50
3,ISF,iShares Core FTSE 100 UCITS ETF,ISF.L,Equity,100,57.41,GBP-dominant,Low,25
4,EQQQ,Invesco EQQQ Nasdaq-100 UCITS ETF,EQQQ.L,Equity,100,57.41,USD,Medium,50
